# 07. Docling + 청킹 — 마크다운을 만들고 검색용 조각으로 자른다

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/08_docling_chunking.ipynb)

> 🧭 **스토리** — 06·07에서 "어떻게 읽느냐"를 봤다. 이제 규정집 전체를 **표가 표로 살아 있는 마크다운**으로 바꾸고, 오후 RAG에서 검색할 **조각(청크)** 으로 자른다. 자르는 방법에 따라 "출장비 표"가 반 토막 나기도 한다.
>
> 📖 **이론 연결** — 이론2 p20~21 (Docling 구조 파싱) · p30~31 (Markdown 변환) · p49~52 (청킹 방식·크기·구조 기반 청킹) · ⏱ **15분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
Docling(MIT)은 레이아웃 모델 → 표 구조 모델(TableFormer) → (필요하면) OCR → 조립을 한 번에 하는 구조 파싱 도구입니다(이론2 p20). 첫 변환 때 레이아웃·표 모델(약 0.5GB)을 받습니다. 설치 시간이 출력됩니다.

In [ ]:
import time
_t0 = time.time()
%pip -q install docling tiktoken
print(f"설치 소요 {time.time() - _t0:.0f}초")

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ 채점 도구(06·07과 같은 셀 일치율) + 정답 청크
import re, pandas as pd
def norm(x):
    return re.sub(r"\s+|\*\*", "", "" if x is None else str(x))

def match_rate(name, rows):
    df = pd.read_csv(fetch(f"tables/{name}.csv"), dtype=str).fillna("")
    gt = [[norm(c) for c in r] for r in [list(df.columns)] + df.values.tolist()]
    rows = [[norm(c) for c in r] for r in (rows or [])]
    if not rows:
        return 0.0
    hit = sum(max(sum(1 for j, c in enumerate(g) if j < len(r) and r[j] == c) for r in rows) for g in gt)
    return hit / sum(len(g) for g in gt)

GT_CHUNKS = load_json("hanbit_chunks_gt.json")
GT_TABLES = {c["table"].split("_")[0]: c["table"] for c in GT_CHUNKS if c["kind"] == "table"}   # 't1' → 't1_domestic'
print("정답 표:", GT_TABLES)

## 1. 디지털 PDF → 마크다운 🔵
코드는 세 줄입니다. `convert()` → `document` → `export_to_markdown()`.

In [ ]:
from docling.document_converter import DocumentConverter
t0 = time.time()
converter = DocumentConverter()
doc = converter.convert(fetch("hanbit_rules.pdf")).document
md = doc.export_to_markdown()
print(f"변환 {time.time() - t0:.0f}초 ({DEVICE}) · 마크다운 {len(md):,}자 · 표 {len(doc.tables)}개\n")
print("\n".join(md.splitlines()[:60]))

💡 **해석** — 강사 PC(CPU)에서 6쪽 변환이 10~20초였습니다(첫 실행은 모델 다운로드 시간이 더해집니다). 장 제목이 `## 제1장 총칙`처럼 **헤딩**이 되고, 표는 `| … |` 마크다운 표가 됩니다. 꼼꼼히 보면 흠도 있습니다.
- 문서 제목 `한빛테크 임직원 출장·복리후생 규정`이 맨 앞이 아니라 **표 0 뒤로 밀려** 있습니다(읽기 순서 오류).
- 무괘선 표 2는 캡션이 `## 표 2. …` **헤딩**으로 바뀌고, 머리글 행(`지역`, `일비 (과장 이하)` …)이 표 밖의 **낱줄 4개**로 떨어졌습니다.

"구조 파싱 도구도 100%는 아니다. 그래서 결과를 정답과 대조한다"가 이 절의 교훈입니다.

## 2. 표 → DataFrame → 정답 대조 🔵
`TableItem.export_to_dataframe(doc=doc)`로 표를 바로 pandas로 꺼냅니다. 정답 CSV 7개와 셀 일치율을 잽니다.

In [ ]:
rows = []
for i, tb in enumerate(doc.tables):
    df = tb.export_to_dataframe(doc=doc)
    gt_name = GT_TABLES.get(f"t{i}", "?")          # 문서 순서 = 표 0~6
    cells = [list(map(str, df.columns))] + df.astype(str).values.tolist()
    rows.append({"표": gt_name, "쪽": tb.prov[0].page_no, "크기": df.shape,
                 "머리글": list(map(str, df.columns))[:3], "셀 일치율": match_rate(gt_name, cells)})
table_report = pd.DataFrame(rows)
display(table_report.style.format({"셀 일치율": "{:.0%}"}))
print("\n[표 2 (무괘선) DataFrame]")
display(doc.tables[2].export_to_dataframe(doc=doc))

💡 **해석** — 강사 환경에서 괘선 표 6개는 모두 **100%**, 무괘선 표 2만 **83%** 였습니다. 표 2의 열 이름이 `0, 1, 2, 3`인 것이 보이죠? 머리글 행이 표 밖으로 빠져서 **숫자만 있고 열의 뜻이 없는 표**가 된 것입니다. 06의 pymupdf4llm과 똑같은 실패입니다. 무괘선 표는 여러 도구가 공통으로 어려워하는 지점이고, 07의 PaddleOCR-VL처럼 **보고 읽는** 방식이 더 나을 수 있습니다.

## 3. 스캔본도 Docling으로 — RapidOCR 한국어 🔵
스캔 이미지(`scan/p2.png`)를 1쪽짜리 PDF로 감싸 Docling에 넣습니다. 글자 층이 없으니 OCR이 필요합니다. Docling 기본 설치에 들어 있는 **RapidOCR**의 언어를 `"korean"`으로 지정합니다(`"ko"`는 이 버전에서 오류가 납니다).

In [ ]:
from PIL import Image
from docling.document_converter import PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions

Image.open(fetch("scan/p2.png")).convert("RGB").save("scan_p2.pdf", resolution=200)   # PNG → 1쪽 PDF
ocr_opts = PdfPipelineOptions(do_ocr=True, ocr_options=RapidOcrOptions(lang=["korean"]))
scan_converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=ocr_opts)})
t0 = time.time()
scan_doc = scan_converter.convert("scan_p2.pdf").document
scan_md = scan_doc.export_to_markdown()
print(f"스캔본 변환 {time.time() - t0:.0f}초\n")
print(scan_md[:1200])

def t1_rate(d):
    best = 0.0
    for tb in d.tables:
        df = tb.export_to_dataframe(doc=d)
        best = max(best, match_rate("t1_domestic", [list(map(str, df.columns))] + df.astype(str).values.tolist()))
    return best
print(f"\n표 1 셀 일치율 — 디지털 PDF {t1_rate(doc):.0%} vs 스캔본+RapidOCR {t1_rate(scan_doc):.0%}")

💡 **해석** — 강사 환경에서 스캔본의 표 1은 **71%** 였습니다(디지털은 100%). 표의 **칸 나누기(구조)와 숫자는 맞았지만**, 머리글이 `(르/) l`처럼 깨지고 직급 칸이 `B`, `12`, 빈칸으로 나왔습니다. 본문도 줄이 통째로 빠지거나 띄어쓰기가 붙습니다(`국내출장`). **같은 Docling이라도 입력이 스캔이면 OCR 엔진의 한국어 실력이 품질을 결정합니다.** 실무에서는 스캔본만 07의 PaddleOCR-VL 같은 한국어 VLM으로 보내는 식으로 나눕니다.

## 4. 청킹 ① 고정 토큰 길이 — 표가 잘리는가? 🔵
가장 단순한 청킹은 **토큰 N개마다 자르기**입니다(겹침 overlap 포함). 오후 RAG가 쓸 `o200k_base` 토큰으로 자르고, **표가 한 청크 안에 온전히 들어 있는지** 검사합니다.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
md_ids = enc.encode(md)
print(f"마크다운 전체 {len(md_ids):,} 토큰")

def fixed_chunks(ids, size, overlap=0):
    step = size - overlap
    return [enc.decode(ids[i:i + size]) for i in range(0, max(len(ids) - overlap, 1), step)]

# 마크다운 안의 표 블록(| 로 시작하는 연속 줄) 7개
TABLE_BLOCKS = re.findall(r"(?:^\|.*\|[ \t]*\n?)+", md, flags=re.M)
def tables_cut(chunks):
    return sum(1 for blk in TABLE_BLOCKS if not any(blk.strip() in c for c in chunks))
def t1_intact(chunks):   # 출장비 표 1: '사원' 행과 '임원' 행이 같은 청크에?
    return any(re.search(r"^\| 사원", c, re.M) and re.search(r"^\| 임원", c, re.M) for c in chunks)

report = []
for size in [200, 400, 800]:
    for ov in [0, 50]:
        ch = fixed_chunks(md_ids, size, ov)
        report.append({"크기(토큰)": size, "겹침": ov, "청크 수": len(ch),
                       f"잘린 표 (/{len(TABLE_BLOCKS)})": tables_cut(ch), "출장비 표 1 온전?": t1_intact(ch)})
display(pd.DataFrame(report))

## 5. 청킹 ② 헤딩(장) 단위 — 표는 통째로 🔵
마크다운의 `## ` 헤딩에서 자릅니다. 장(章) 하나가 한 청크가 되고, 표는 캡션과 함께 **절대 잘리지 않습니다**. 대신 청크 길이가 들쭉날쭉합니다.

In [ ]:
import matplotlib.pyplot as plt
heading_chunks = [c.strip() for c in re.split(r"(?m)^(?=## )", md) if c.strip()]
print(f"헤딩 청크 {len(heading_chunks)}개 · 잘린 표 {tables_cut(heading_chunks)}/{len(TABLE_BLOCKS)} · 출장비 표 1 온전? {t1_intact(heading_chunks)}")
for c in heading_chunks:
    print(f"  {len(enc.encode(c)):>4} 토큰 | {c.splitlines()[0][:40]}")

fixed400 = fixed_chunks(md_ids, 400, 50)
plt.figure(figsize=(7, 3))
plt.hist([len(enc.encode(c)) for c in fixed400], bins=12, alpha=0.6, label=f"고정 400(겹침 50) · {len(fixed400)}개")
plt.hist([len(enc.encode(c)) for c in heading_chunks], bins=12, alpha=0.6, label=f"헤딩 단위 · {len(heading_chunks)}개")
plt.xlabel("청크 길이(토큰)"); plt.ylabel("청크 수"); plt.title("청크 길이 분포"); plt.legend(); plt.tight_layout(); plt.show()

💡 **해석** — 4절 표에서 고정 길이 청킹은 크기·겹침을 바꿔도 **표가 여러 개씩 잘립니다**. 경계가 의미가 아니라 토큰 개수로 정해지기 때문입니다. 출장비 표 1이 반으로 잘리면 "임원 숙박비 상한?"을 물었을 때 **머리글 없는 숫자 조각**만 검색되어 틀린 답이 나옵니다.
헤딩 단위는 잘린 표가 **0개**지만, 길이 분포를 보면 짧은 조각(문서 제목·표 2 헤딩)과 긴 조각(표 2개가 든 장)이 섞여 있습니다. 너무 긴 청크는 검색 정밀도를 떨어뜨리고, 너무 짧은 청크는 맥락이 없습니다(이론2 p50). 그래서 다음 절처럼 **구조 + 길이 상한**을 함께 쓰는 방법이 나옵니다.

## 6. 청크 JSON 저장 — 섹션·페이지 메타데이터 🔵
오후 노트북(08·10·12)이 쓸 청크를 정답과 **같은 스키마**(`id, section, section_title, page, kind, table, text`)로 만듭니다. Docling 문서의 항목을 순서대로 돌면서 **장 본문은 한 청크, 표는 캡션을 붙여 따로 한 청크**로 묶습니다. 페이지 번호는 `prov[0].page_no`에서 옵니다.

In [ ]:
from docling_core.types.doc import TableItem, DocItemLabel
mine, sec, last_caption = [], None, ""
for item, _level in doc.iterate_items():
    text = getattr(item, "text", "") or ""
    page = item.prov[0].page_no if item.prov else None
    chap = re.match(r"제(\d+)장", text) if item.label == DocItemLabel.SECTION_HEADER else None
    if chap:                                                    # 새 장 시작 → 본문 청크 하나 열기
        sec = {"id": f"s{chap.group(1)}_text", "section": chap.group(1), "section_title": text,
               "page": page, "kind": "text", "text": text}
        mine.append(sec)
    elif isinstance(item, TableItem):                           # 표 → 캡션을 붙인 별도 청크
        cap = item.caption_text(doc) or last_caption
        num = re.search(r"표 (\d+)", cap)
        tname = GT_TABLES.get(f"t{num.group(1)}" if num else "", "unknown")
        tbl_md = item.export_to_markdown(doc=doc).replace(cap, "", 1).strip()   # 표 마크다운에 붙어 나오는 캡션은 한 번만
        mine.append({"id": f"s{sec['section']}_{tname}", "section": sec["section"], "section_title": sec["section_title"],
                     "page": page, "kind": "table", "table": tname,
                     "text": f"{sec['section_title']} — {cap}\n\n{tbl_md}"})
    elif item.label == DocItemLabel.CAPTION:                    # 캡션은 표 청크에 이미 들어감
        continue
    elif re.match(r"표 \d+\.", text):                          # 헤딩으로 잘못 잡힌 캡션(표 2) 기억
        last_caption = text
    elif sec is not None and text:
        sec["text"] += "\n\n" + text

with open("hanbit_chunks_mine.json", "w", encoding="utf-8") as f:
    json.dump(mine, f, ensure_ascii=False, indent=1)

import difflib
gt_by_id = {c["id"]: c for c in GT_CHUNKS}
def sim(a, b):   # 공백·구분선(---) 차이는 무시하고 비교
    squash = lambda t: re.sub(r"[\s\-*]", "", t)
    return difflib.SequenceMatcher(None, squash(a), squash(b)).ratio()
cmp = pd.DataFrame([{"id": c["id"], "kind": c["kind"], "page": c["page"],
                     "정답과 텍스트 유사도": sim(c["text"], gt_by_id[c["id"]]["text"])
                                          if c["id"] in gt_by_id else None} for c in mine])
print(f"내 청크 {len(mine)}개 · 정답 {len(GT_CHUNKS)}개 · id 일치 {len(set(cmp['id']) & set(gt_by_id))}개 → hanbit_chunks_mine.json 저장")
display(cmp.style.format({"정답과 텍스트 유사도": "{:.2f}"}))

💡 **해석** — 강사 환경에서 13개 청크의 id가 정답과 모두 일치했습니다. 텍스트 유사도가 1.0보다 낮은 곳은 세 군데(1장 본문 0.93 · 3장 본문 0.94 · 표 2 0.82)였습니다. 1장 본문에는 표 0 뒤로 밀린 **문서 제목**이, 3장 본문에는 표에서 떨어져 나온 **표 2 머리글 낱줄**이 섞여 들어갔고, 표 2 청크에는 그 머리글 행이 없습니다. 파싱 오류는 청크와 검색까지 그대로 전파됩니다. 오후 실습은 이 정답 청크(`hanbit_chunks_gt.json`)를 씁니다.

## 7. 심화 🟠 Docling HybridChunker — 구조 + 토큰 상한
`HybridChunker`는 문서 구조(헤딩·표·목록)를 따라 자르되, **토큰 상한을 넘는 조각만** 더 쪼개고 너무 작은 이웃은 합칩니다. 토크나이저로 `tiktoken`의 `o200k_base`를 넣어 4절과 같은 기준으로 셉니다. `contextualize()`는 청크 앞에 **헤딩 경로**를 붙여 임베딩용 텍스트를 만듭니다.

In [ ]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer

MAX_TOKENS = 400                                         # ← 200, 800 으로도 바꿔 보세요
chunker = HybridChunker(tokenizer=OpenAITokenizer(tokenizer=enc, max_tokens=MAX_TOKENS))
hybrid = list(chunker.chunk(doc))
lens = [len(enc.encode(chunker.contextualize(c))) for c in hybrid]
print(f"HybridChunker(max_tokens={MAX_TOKENS}) → {len(hybrid)}개 · 길이 {min(lens)}~{max(lens)} 토큰\n")
for c in hybrid[:3]:
    pages = sorted({p.page_no for it in c.meta.doc_items for p in it.prov})
    print(f"--- headings={c.meta.headings} · pages={pages}")
    print(chunker.contextualize(c)[:300], "\n")

💡 **해석** — 강사 환경에서 규정집 전체가 **14개 청크, 46~330 토큰**으로 나뉘었습니다. 표 0 청크가 `1.0, 시행일 = 2024-01-01. 1.0, 주요 변경 내용 = 규정 제정. …`처럼 **"행 머리, 열 이름 = 값" 문장**으로 바뀌어 나왔습니다. 각 숫자 옆에 열 이름이 붙어 있어서, 표를 잘라도 조각 하나하나가 **스스로 뜻을 가집니다**. 길이 상한도 지켜집니다. 구조 청킹과 고정 길이 청킹의 장점을 합친 방식입니다.

## 8. 바꿔서 실행 🟡
1. **표가 안 잘리는 최소 고정 크기 찾기** — `OVERLAP`을 바꿔 가며, 7개 표가 하나도 안 잘리는 가장 작은 청크 크기를 찾아보세요.
2. **개인정보 마스킹** — 규정집에는 없지만 실제 사내 문서(신청서·메일)에는 전화번호·이메일·주민등록번호가 섞여 있습니다. 청크로 저장하기 **전에** 지워야 합니다. 정규식을 고쳐 `samples`의 세 문장을 모두 가려 보세요.

In [ ]:
OVERLAP = 0                                              # ← 50, 100 으로 바꿔 보세요
found = None
for size in range(100, 1601, 50):
    if tables_cut(fixed_chunks(md_ids, size, OVERLAP)) == 0:
        found = size; break
print(f"겹침 {OVERLAP}: 표 7개가 모두 온전한 최소 크기 = {found} 토큰" if found else
      f"겹침 {OVERLAP}: 1,600 토큰까지 찾지 못함 — 고정 길이로는 '운'에 맡겨야 합니다")
print(f"참고: 문서 전체 {len(md_ids):,} 토큰 · 가장 긴 표 {max(len(enc.encode(b)) for b in TABLE_BLOCKS)} 토큰")

In [ ]:
samples = [
    "출장 문의는 김한빛 대리(010-1234-5678)에게 연락 바랍니다.",
    "증빙 서류는 hr.team@hanbit-tech.example 로 보내 주세요.",
    "경조금 신청서: 이름 박규정, 주민등록번호 900101-1234567, 계좌 확인 요망",
]
PII = {                                                   # ← 패턴을 고치거나 추가해 보세요
    "전화": r"01[016789]-?\d{3,4}-?\d{4}",
    "이메일": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "주민번호": r"\d{6}-?[1-4]\d{6}",
}
def mask(s):
    for tag, pat in PII.items():
        s = re.sub(pat, f"[{tag}]", s)
    return s
for s in samples:
    print(mask(s))

### 🤔 생각해보기

청크를 **작게 자를수록 검색 정밀도가 오르는데**, 왜 무한정 작게 자를 수 없을까요? 이 규정집에서 "청크 하나"는 어떤 단위가 적당할까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **맥락이 사라진다**: 50토큰짜리 조각 `| 차장 | 35,000 | 35,000 | 120,000 |`에는 "무슨 표의 어느 열인지"가 없습니다. 검색에 걸려도 LLM이 답을 만들 수 없습니다(머리글·캡션·장 제목이 필요).
- **검색 대상이 늘어난다**: 청크 수가 늘면 임베딩 비용·저장 공간·검색 시간이 늘고, 비슷한 조각끼리 순위 경쟁이 심해져 top-k 안에 정답이 덜 들어옵니다.
- **답에 필요한 근거가 흩어진다**: 조건(본문 제17조)과 금액(표 1)이 다른 조각에 있으면 둘 다 가져와야 합니다.
- 이 규정집이라면 **"장 본문 하나 + 표 하나(캡션·장 제목 포함)"** 가 자연스러운 단위입니다. 정답 청크(`hanbit_chunks_gt.json`)가 그렇게 만들어져 있습니다. 문서가 길어지면 HybridChunker처럼 구조를 따르되 상한(예: 400토큰)을 두고, 표 조각마다 머리글을 반복해 붙입니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- Docling은 레이아웃 → 표 구조 → OCR → 조립을 한 번에 해서 표를 DataFrame·마크다운으로 준다. 무괘선 표·읽기 순서는 여전히 검수 대상.
- 스캔본은 같은 Docling이라도 OCR 엔진의 한국어 실력이 품질을 정한다(RapidOCR은 `lang=["korean"]`).
- 고정 토큰 청킹은 표를 자른다. 헤딩·표 단위 청킹은 안 자르지만 길이가 들쭉날쭉하다 → HybridChunker.
- 청크에는 섹션·페이지·종류 메타데이터를 붙여 저장한다. 개인정보는 저장 전에 마스킹.

> 다음 노트북: **09_text_embeddings_compare — 청크를 벡터로 바꾸고 한국어 임베딩 모델을 비교한다**